In [1]:
class MyStack:
    def pop(self):
        return "Removed"

def remove_last(obj):
    return obj.pop()

print(remove_last([1, 2, 3]))  # 3
print(remove_last(MyStack()))  # Removed

3
Removed


In [2]:
class MySequence:
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return self.data[index]

seq = MySequence([10, 20, 30])

print(len(seq))       # 3
print(seq[1])         # 20

for item in seq:
    print(item)

3
20
10
20
30


In [6]:
class Foo:
  def __getitem__(self, pos):
    return range(0, 30, 10)[pos]

In [8]:
PyObject *
PyObject_GetIter(PyObject *o)
{
 {
    PyTypeObject *t = Py_TYPE(o);
    getiterfunc f;

    f = t->tp_iter;
    if (f == NULL) {
        if (PySequence_Check(o))
            return PySeqIter_New(o);
        return type_error("'%.200s' object is not iterable", o);
    }
    else {
        PyObject *res = (*f)(o);
        if (res != NULL && !PyIter_Check(res)) {
            PyErr_Format(PyExc_TypeError,
                         "%T.__iter__() must return an iterator, not %T",
                         o, res);
            Py_SETREF(res, NULL);
        }
        return res;
    }
}

SyntaxError: invalid syntax (1710247812.py, line 1)

In [9]:
import collections

Card = collections.namedtuple('Card', ['rank', 'suit'])

class FrenchDeck:
    ranks = [str(n) for n in range(2, 11)] + list('JQKA')
    suits = 'spades diamonds clubs hearts'.split()

    def __init__(self):
        self._cards = [Card(rank, suit) for suit in self.suits
                                        for rank in self.ranks]

    def __len__(self):
        return len(self._cards)

    def __getitem__(self, position):
        return self._cards[position]

In [11]:
>>> from random import shuffle
>>> from frenchdeck import FrenchDeck
>>> deck = FrenchDeck()
>>> shuffle(deck)

ModuleNotFoundError: No module named 'frenchdeck'

In [12]:
>>> def set_card(deck, position, card):      # ①
...     deck._cards[position] = card
...
>>> FrenchDeck.__setitem__ = set_card        # ②
>>> shuffle(deck)                            # ③
>>> deck[:5]
[Card(rank='3', suit='hearts'), Card(rank='4', suit='diamonds'), Card(rank='4', suit='clubs'), Card(rank='7', suit='hearts'), Card(rank='9', suit='spades')]

NameError: name 'deck' is not defined

In [13]:
import collections

Card = collections.namedtuple('Card', ['rank', 'suit'])

class FrenchDeck2(collections.MutableSequence):
    ranks = [str(n) for n in range(2, 11)] + list('JQKA')
    suits = 'spades diamonds clubs hearts'.split()

    def __init__(self):
        self._cards = [Card(rank, suit) for suit in self.suits
                                        for rank in self.ranks]

    def __len__(self):
        return len(self._cards)

    def __getitem__(self, position):
        return self._cards[position]

    def __setitem__(self, position, value):   # ①
        self._cards[position] = value

    def __delitem__(self, position):          # ②
        del self._cards[position]

    def insert(self, position, value):        # ③
        self._cards.insert(position, value)

AttributeError: module 'collections' has no attribute 'MutableSequence'

In [14]:
from abc import ABC, abstractmethod

class Drawable(ABC):
    @abstractmethod
    def draw(self):
        pass
class ExternalPainter:
    def draw(self):
        print("Drawing")

Drawable.register(ExternalPainter)

obj = ExternalPainter()

print(isinstance(obj, Drawable))  # True

True


In [ ]:
from collections.abc import Sized

class MyCollection:
    def __len__(self):
        return 10

Sized.register(MyCollection)

In [ ]:
import abc

class Tombola(abc.ABC):

    @abc.abstractmethod
    def load(self, iterable):

    @abc.abstractmethod
    def pick(self):

    def loaded(self):
        return bool(self.inspect())

    def inspect(self):
        items = []
        while True:
            try:
                items.append(self.pick())
            except LookupError:
                break
        self.load(items)
        return tuple(sorted(items))

In [ ]:
import random

from tombola import Tombola

class BingoCage(Tombola):

    def __init__(self, items):
        self._randomizer = random.SystemRandom()
        self._items = []
        self.load(items)

    def load(self, items):
        self._items.extend(items)
        self._randomizer.shuffle(self._items)

    def pick(self):
        try:
            return self._items.pop()
        except IndexError:
            raise LookupError('pick from empty BingoCage')

    def __call__(self):
        self.pick()

In [ ]:
import random

from tombola import Tombola

class LotteryBlower(Tombola):

    def __init__(self, iterable):
        self._balls = list(iterable)                     # ①

    def load(self, iterable):
        self._balls.extend(iterable)

    def pick(self):
        try:
            position = random.randrange(len(self._balls))    # ②
        except ValueError:
            raise LookupError('pick from empty BingoCage')
        return self._balls.pop(position)                 # ③

    def loaded(self):                                    # ④
        return bool(self._balls)

    def inspect(self):                                   # ⑤
        return tuple(sorted(self._balls))

In [ ]:
from random import randrange

from tombola import Tombola

@Tombola.register                                        # ①
class TomboList(list):                                   # ②

    def pick(self):
        if self:                                         # ③
            position = randrange(len(self))
            return self.pop(position)                    # ④
        else:
            raise LookupError('pop from empty TomboList')

    load = list.extend                                   # ⑤

    def loaded(self):
        return bool(self)                                # ⑥

    def inspect(self):
        return tuple(sorted(self))

In [ ]:
import doctest

from tombola import Tombola

# modules to test
import bingo, lotto, tombolist, drum                     # ①

TEST_FILE = 'tombola_tests.rst'
TEST_MSG = '{0:16} {1.attempted:2} tests, {1.failed:2} failed - {2}'


def main(argv):
    verbose = '-v' in argv
    real_subclasses = Tombola.__subclasses__()           # ②
    virtual_subclasses = list(Tombola._abc_registry)     # ③

    for cls in real_subclasses + virtual_subclasses:     # ④
        test(cls, verbose)


def test(cls, verbose=False):

    res = doctest.testfile(
            TEST_FILE,
            globs={'ConcreteTombola': cls},              # ⑤
            verbose=verbose,
            optionflags=doctest.REPORT_ONLY_FIRST_FAILURE)
    tag = 'FAIL' if res.failed else 'OK'
    print(TEST_MSG.format(cls.__name__, res, tag))       # ⑥


if __name__ == '__main__':
    import sys
    main(sys.argv)